<a href="https://colab.research.google.com/github/RakePants/nerdless/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://huggingface.co/transformers/v3.2.0/custom_datasets.html

In [ ]:
!pip install transformers -U

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 85.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.4/182.4 KB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 94.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("drive/MyDrive/nerdless/chan_dialogues_scored_tox.csv", header=None, on_bad_lines='skip', engine="python").applymap(str)
data.head()

,0,1
0,"Себорея, витилиго, нейродермит, псориаз и прочие.",От псориаза можно умереть? Пиздец страшно то к...
1,в это говно только такие говноеды как и играли,Блядь
2,С модами играл или на сухую?,В ваниллу
3,На четвёртый. Потому что с 3 героями плохие во...,Это тебя у нас в компьютерном клубе в жопу вые...
4,"Нагибатор Player'ов, артуров и демонов нашелся...",+ с адскими пингами.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Kirili4ik/ruDialoGpt3-medium-finetuned-telegram"  
tokenizer =  AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)

Downloading:   0%|          | 0.00/608 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.98M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/275 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/947 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

In [ ]:
model = model.to('cuda')

In [ ]:
#sample_data = ["I am eating","I am playing "]
#tokenizer(sample_data, padding=True, truncation=True, max_length=512)

In [ ]:
X = list(data.iloc[:100000, 0])
y = list(data.iloc[:100000, 1])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
X_train_tokenized = tokenizer(X_train, padding=True, truncation=True, max_length=64)
X_val_tokenized = tokenizer(X_val, padding=True, truncation=True, max_length=64)
y_train_tokenized = tokenizer(y_train, padding=True, truncation=True, max_length=64)
y_val_tokenized = tokenizer(y_val, padding=True, truncation=True, max_length=64)

In [ ]:
X_train_tokenized.keys()

dict_keys(['input_ids', 'attention_mask'])

In [ ]:
len(X_train),len(X_val)

(80000, 20000)

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets
    
    def __len__(self):
        return len(self.targets["input_ids"])
    
    def __getitem__(self, index):
        input_ids = torch.tensor(self.inputs["input_ids"][index])#.squeeze()
        target_ids = torch.tensor(self.targets["input_ids"][index])#.squeeze()
        
        return {"input_ids": input_ids, "labels": target_ids}

In [ ]:
train_dataset = Dataset(X_train_tokenized, y_train_tokenized)
val_dataset = Dataset(X_val_tokenized, y_val_tokenized)

In [ ]:
train_dataset[1]

{'input_ids': tensor([5786,  343,  345,  357,  309, 3910,  295, 6934,   18,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]),
 'labels': tensor([ 5786,  7425,   357,   281,  3136, 23642,    16,  3291,   864, 39249,
           352,   807,   539,    18,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0])}

In [ ]:
def compute_metrics(p):
    print(type(p))
    pred, labels = p
    pred = np.argmax(pred, axis=1)

    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    recall = recall_score(y_true=labels, y_pred=pred)
    precision = precision_score(y_true=labels, y_pred=pred)
    f1 = f1_score(y_true=labels, y_pred=pred)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# Define Trainer
args = TrainingArguments(
    output_dir="output",
    num_train_epochs=1,
    per_device_train_batch_size=16
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [ ]:
trainer.train()

***** Running training *****
  Num examples = 80000
  Num Epochs = 1
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 10000
  Number of trainable parameters = 355871744


Step,Training Loss
500,1.454800
1000,1.256900
1500,1.272200
2000,1.264300
2500,1.270300
3000,1.263200
3500,1.255300
4000,1.225500
4500,1.250200
5000,1.248400


Saving model checkpoint to output/checkpoint-500
Configuration saved in output/checkpoint-500/config.json
Model weights saved in output/checkpoint-500/pytorch_model.bin
Saving model checkpoint to output/checkpoint-1000
Configuration saved in output/checkpoint-1000/config.json
Model weights saved in output/checkpoint-1000/pytorch_model.bin
Saving model checkpoint to output/checkpoint-1500
Configuration saved in output/checkpoint-1500/config.json
Model weights saved in output/checkpoint-1500/pytorch_model.bin
Saving model checkpoint to output/checkpoint-2000
Configuration saved in output/checkpoint-2000/config.json
Model weights saved in output/checkpoint-2000/pytorch_model.bin
Saving model checkpoint to output/checkpoint-2500
Configuration saved in output/checkpoint-2500/config.json
Model weights saved in output/checkpoint-2500/pytorch_model.bin
Saving model checkpoint to output/checkpoint-3000
Configuration saved in output/checkpoint-3000/config.json
Model weights saved in output/check

RuntimeError: ignored

In [ ]:
trainer.evaluate()

***** Running Evaluation *****
  Num examples = 20000
  Batch size = 8


ValueError: ignored

In [ ]:
np.set_printoptions(suppress=True)

In [ ]:
checkpoint = "Kirili4ik/ruDialoGpt3-medium-finetuned-telegram"   
tokenizer =  AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained("/content/output/checkpoint-5500")
model.eval()


# util function to get expected len after tokenizing
def get_length_param(text: str, tokenizer) -> str:
    tokens_count = len(tokenizer.encode(text))
    if tokens_count <= 15:
        len_param = '1'
    elif tokens_count <= 50:
        len_param = '2'
    elif tokens_count <= 256:
        len_param = '3'
    else:
        len_param = '-'
    return len_param


# util function to get next person number (1/0) for Machine or Human in the dialogue
def get_user_param(text: dict, machine_name_in_chat: str) -> str:
    if text['from'] == machine_name_in_chat:
        return '1'  # machine
    else:
        return '0'  # human


chat_history_ids = torch.zeros((1, 0), dtype=torch.int)

while True:
    
    next_who = input("Who's phrase?\t")  #input("H / G?")     # Human or GPT

    # In case Human
    if next_who == "H" or next_who == "Human":
        input_user = input("===> Human: ")
        
        # encode the new user input, add parameters and return a tensor in Pytorch
        new_user_input_ids = tokenizer.encode(f"|0|{get_length_param(input_user, tokenizer)}|" \
                                              + input_user + tokenizer.eos_token, return_tensors="pt")
        # append the new user input tokens to the chat history
        chat_history_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)

    if next_who == "G" or next_who == "GPT":

        next_len = input("Phrase len? 1/2/3/-\t")  #input("Exp. len?(-/1/2/3): ")
        # encode the new user input, add parameters and return a tensor in Pytorch
        new_user_input_ids = tokenizer.encode(f"|1|{next_len}|", return_tensors="pt")
        # append the new user input tokens to the chat history
        chat_history_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        
        # print(tokenizer.decode(chat_history_ids[-1])) # uncomment to see full gpt input
        
        # save previous len
        input_len = chat_history_ids.shape[-1]
        # generated a response; PS you can read about the parameters at hf.co/blog/how-to-generate
        chat_history_ids = model.generate(
            chat_history_ids,
            num_return_sequences=1,                     # use for more variants, but have to print [i]
            max_length=512,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature = 0.6,                          # 0 for greedy
            
            eos_token_id=tokenizer.eos_token_id,
            
            pad_token_id=tokenizer.pad_token_id,
            
        )
        
        
        # pretty print last ouput tokens from bot
        print(f"===> GPT-3:  {tokenizer.decode(chat_history_ids[:, input_len:][0], skip_special_tokens=True)}")

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Kirili4ik--ruDialoGpt3-medium-finetuned-telegram/snapshots/6625aee6169f5087a7dccc47ded8906c44478ecf/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Kirili4ik--ruDialoGpt3-medium-finetuned-telegram/snapshots/6625aee6169f5087a7dccc47ded8906c44478ecf/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Kirili4ik--ruDialoGpt3-medium-finetuned-telegram/snapshots/6625aee6169f5087a7dccc47ded8906c44478ecf/tokenizer.json
loading file added_tokens.json from cache at /root/.cache/huggingface/hub/models--Kirili4ik--ruDialoGpt3-medium-finetuned-telegram/snapshots/6625aee6169f5087a7dccc47ded8906c44478ecf/added_tokens.json
loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--Kirili4ik--ruDialoGpt3-medium-finetuned-telegram/snapshots/6625aee6169f5087a7dccc47ded8906c44478ecf/special_tokens_map.json
loading file tokenizer

Who's phrase?	H
===> Human: привет
Who's phrase?	G
Phrase len? 1/2/3/-	1
===> GPT-3:  .?,УБЬЮЛ!ЕХ @!!!E11�ЫАА�ЙРААН0 СДwВАNКА Б�!!А1ЕТТЬЯ-бР3UA/ А на) УЧКУ неDЁ МТ в ЕXШН ПОВLBcomМУOБУМЖoutube Т ОЛАДЕKДУЛЯ"5�FsХАV�C ВНИЗ8СКp10ЛИW НАПВО�НЫЙ ЗАv КОHЩ К ПiMГО7PТУЕЙ................Q_T?!b ЙНОРО ЯЛЬ ИI2АТЬМИ ЧБАСЬ...�СЯйДА ТЕy МОМА ОБo НЕkИМО ДА ПРО9thПА= БЫ Ё БУИТ6СТО@ЛЕныйЦ
Who's phrase?	H
===> Human: але ты где
Who's phrase?	G
Phrase len? 1/2/3/-	-


Input length of input_ids is 526, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


===> GPT-3:  
Who's phrase?	H
===> Human: але ты где
Who's phrase?	G
Phrase len? 1/2/3/-	3


Input length of input_ids is 541, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


===> GPT-3:  
Who's phrase?	H
===> Human: але ты где
Who's phrase?	G
Phrase len? 1/2/3/-	2


Input length of input_ids is 556, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


===> GPT-3:  
Who's phrase?	H
===> Human: але ты где
Who's phrase?	G
Phrase len? 1/2/3/-	1


Input length of input_ids is 571, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


===> GPT-3:  ?


In [ ]:
text = "That was good point"
# text = "go to hell"
inputs = tokenizer(text, padding = True, truncation = True, return_tensors='pt').to('cuda')
outputs = model(**inputs)
print(outputs)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)
predictions = predictions.cpu().detach().numpy()
predictions

SequenceClassifierOutput(loss=None, logits=tensor([[ 1.4725, -2.1523]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[0.9740, 0.0260]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


array([[0.9740369 , 0.02596313]], dtype=float32)

In [ ]:
trainer.save_model('CustomModel')

Saving model checkpoint to CustomModel
Configuration saved in CustomModel/config.json
Model weights saved in CustomModel/pytorch_model.bin


In [ ]:
# trainer.save_model('/content/drive/MyDrive/Youtube Tutorials/toxic')
# model_2 = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/Youtube Tutorials/toxic")
# model_2.to('cuda')

In [ ]:
model_2 = BertForSequenceClassification.from_pretrained("CustomModel")
model_2.to('cuda')

In [ ]:
# text = "That was good point"
text = "go to hell"
inputs = tokenizer(text,padding = True, truncation = True, return_tensors='pt').to('cuda')
outputs = model_2(**inputs)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
predictions = predictions.cpu().detach().numpy()
predictions

array([[0.00004367, 0.99995637]], dtype=float32)